# 00 - Database & SQL
CareFlow | Yash Rahangdale | Infotact Solutions

Yahan hum event log ko SQL (SQLite demo / MySQL / PostgreSQL) me load karenge aur KPI queries chalayenge.

In [ ]:
import sys, pandas as pd
sys.path.append('../src')
from connect_db import get_engine, build_sqlite, run_kpis
eng = get_engine('sqlite')
build_sqlite(eng)

## Overall KPI

In [ ]:
pd.read_sql('SELECT COUNT(*) cases, ROUND(AVG(los_hours),2) avg_los_h, ROUND(SUM(total_cost_inr),0) total_cost FROM fact_case', eng)

## Bottleneck: activity-wise waiting time (pure SQL)

In [ ]:
pd.read_sql('''SELECT activity, COUNT(*) occurrences, ROUND(AVG(wait_minutes),1) avg_wait_min,
 ROUND(AVG(service_minutes),1) avg_service_min FROM fact_event GROUP BY activity ORDER BY avg_wait_min DESC''', eng)

## Directly-Follows edges (LEAD window function)

In [ ]:
pd.read_sql('''SELECT activity, next_activity, COUNT(*) edge_count FROM (SELECT case_id, event_index, activity,
 LEAD(activity) OVER (PARTITION BY case_id ORDER BY event_index) next_activity FROM fact_event)
 WHERE next_activity IS NOT NULL GROUP BY activity, next_activity HAVING COUNT(*)>=3 ORDER BY edge_count DESC LIMIT 10''', eng)